# Airbnb Paris – Cleaned
- Numerische/encodierte Features (Frequency-Encoding + OHE), Freitexte entfernt, skaliert
- ICC-Label `is_top_rating` (rating==5 → 1, rating<=3 → 0); `row_id` als Join-Key
- Eine CSV ohne Split; Transforms auf dem gesamten Datensatz gefittet

In [9]:
import re
import unicodedata
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

RAW_PATH = "../../data/raw/airbnb_paris.csv"
OUT_PATH = "../../data/preprocessed/cleaned_airbnb_paris.csv"

## Rohdaten laden
- `row_id` = `id` (stabiler Join-Key für `cleaned_text`/Splits)

In [10]:
df = pd.read_csv(RAW_PATH, low_memory=False)
df["row_id"] = df["id"]
print("Rohdaten-Shape:", df.shape)

Rohdaten-Shape: (81853, 80)


## Nicht-relevante / leere Spalten entfernen
- IDs/URLs/Scrape-Meta, Target-Sub-Scores (Leakage), Review-Counts/-Daten, Kalender-Meta, Duplikate, 100%-NaN (price/beds-Quellen)

In [11]:
columns_to_drop = [
    "id", "listing_url", "scrape_id", "last_scraped", "source", "picture_url",
    "host_id", "host_url", "host_name", "host_thumbnail_url", "host_picture_url", "license",
    "review_scores_accuracy", "review_scores_cleanliness", "review_scores_checkin",
    "review_scores_communication", "review_scores_location", "review_scores_value",
    "number_of_reviews", "number_of_reviews_ltm", "number_of_reviews_l30d",
    "first_review", "last_review", "reviews_per_month",
    "calendar_updated", "calendar_last_scraped", "bathrooms_text",
    "host_listings_count", "host_total_listings_count",
    "minimum_minimum_nights", "maximum_minimum_nights",
    "minimum_maximum_nights", "maximum_maximum_nights",
    "minimum_nights_avg_ntm", "maximum_nights_avg_ntm",
    "has_availability", "host_neighbourhood", "neighbourhood",
    "price", "estimated_revenue_l365d", "neighbourhood_group_cleansed",
]
df = df.drop(columns=[c for c in columns_to_drop if c in df.columns])
print("Nach Drop:", df.shape)

Nach Drop: (81853, 39)


## ICC-Target erstellen
- Inlier (`is_top_rating == 1`): rating == 5; Outlier (0): rating <= 3; Mittelfeld/NaN verworfen

In [12]:
df = df.dropna(subset=["review_scores_rating"]).copy()
df = df.loc[(df["review_scores_rating"] == 5.0) | (df["review_scores_rating"] <= 3.0)].copy()
df["is_top_rating"] = (df["review_scores_rating"] == 5.0).astype(int)
print("Nach Target:", df.shape, "| Outlier-Rate:", round((df["is_top_rating"] == 0).mean(), 4))

Nach Target: (18350, 40) | Outlier-Rate: 0.0393


## Feature Engineering
- host_tenure_days, Prozent-Raten → float, Listen-Counts, host_location-Flags, Booleans t/f → 0/1

In [13]:
df["host_since"] = pd.to_datetime(df["host_since"], errors="coerce")
df["host_tenure_days"] = (df["host_since"].max() - df["host_since"]).dt.days
df = df.drop(columns=["host_since"])

df["host_response_rate"] = pd.to_numeric(df["host_response_rate"].astype(str).str.rstrip("%"), errors="coerce")
df["host_acceptance_rate"] = pd.to_numeric(df["host_acceptance_rate"].astype(str).str.rstrip("%"), errors="coerce")

df["amenities_count"] = df["amenities"].fillna("[]").apply(lambda s: s.count(",") + 1 if len(s) > 2 else 0)
df["host_verifications_count"] = df["host_verifications"].fillna("[]").apply(lambda s: s.count(",") + 1 if len(s) > 2 else 0)
df = df.drop(columns=["amenities", "host_verifications"])

loc = df["host_location"].fillna("")
df["host_in_paris"] = loc.str.contains("Paris", case=False, na=False).astype(int)
df["host_in_france"] = loc.str.contains("France", case=False, na=False).astype(int)
df["host_location_missing"] = df["host_location"].isna().astype(int)
df = df.drop(columns=["host_location"])

for c in ["host_is_superhost", "host_has_profile_pic", "host_identity_verified", "instant_bookable"]:
    df[c] = df[c].map({"t": 1, "f": 0})
print("Nach FE:", df.shape)

Nach FE: (18350, 42)


## Imputation & Konstanten-Drop
- Numerisch: Median; Boolean: Mode; Kategorisch: "unknown". Konstante numerische Spalten (z. B. price/beds-Reste) droppen

In [14]:
text_cols = ["name", "description", "neighborhood_overview", "host_about"]
categorical_cols = ["host_response_time", "neighbourhood_cleansed", "property_type", "room_type"]
bool_cols = ["host_is_superhost", "host_has_profile_pic", "host_identity_verified", "instant_bookable",
             "host_in_paris", "host_in_france", "host_location_missing"]
non_numeric = set(text_cols + categorical_cols + bool_cols + ["is_top_rating", "review_scores_rating", "row_id"])

for c in [c for c in df.columns if c not in non_numeric]:
    df[c] = df[c].fillna(df[c].median())
for c in bool_cols:
    if df[c].isna().any():
        df[c] = df[c].fillna(df[c].mode().iloc[0])
for c in categorical_cols:
    df[c] = df[c].fillna("unknown")

constant_cols = [c for c in df.columns
                 if c not in text_cols and pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique(dropna=False) <= 1]
df = df.drop(columns=constant_cols)
print("Konstante gedroppt:", constant_cols)

Konstante gedroppt: ['bathrooms', 'beds']


/home/debian/TFM_master_thesis/.venv/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/debian/TFM_master_thesis/.venv/lib/python3.14/site-packages/numpy/lib/_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


## Encoding & Skalierung
- Frequency-Encoding: neighbourhood_cleansed, property_type; OHE: host_response_time, room_type
- StandardScaler auf numerische + freq-Spalten (Boolean/OHE/Target/row_id ausgenommen)

In [15]:
ohe_cols = ["host_response_time", "room_type"]
freq_cols = ["neighbourhood_cleansed", "property_type"]

for c in freq_cols:
    df[c + "_freq"] = df[c].map(df[c].value_counts(normalize=True))
df = df.drop(columns=freq_cols)
df = pd.get_dummies(df, columns=ohe_cols, dummy_na=False, dtype=int)

exclude = set(text_cols + bool_cols + ["is_top_rating", "review_scores_rating", "row_id"])
exclude.update([c for c in df.columns if any(c.startswith(p + "_") for p in ohe_cols)])
scale_cols = [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]
df[scale_cols] = StandardScaler().fit_transform(df[scale_cols])
print("Skaliert:", len(scale_cols), "Spalten")

Skaliert: 24 Spalten


## Spaltennamen normalisieren & speichern
- snake_case; `review_scores_rating` + Freitexte entfernen (numerisch only); `row_id` bleibt

In [16]:
df.columns = [re.sub(r"_+", "_", re.sub(r"[^a-zA-Z0-9]+", "_",
              unicodedata.normalize("NFKD", c).encode("ascii", "ignore").decode("ascii")).lower()).strip("_")
              for c in df.columns]
df = df.drop(columns=["review_scores_rating"] + text_cols)
df.to_csv(OUT_PATH, index=False)

print("Gespeichert:", OUT_PATH, "| Shape:", df.shape)
print("row_id unique:", df["row_id"].is_unique, "| NaN:", int(df.isna().sum().sum()))

Gespeichert: ../../data/preprocessed/cleaned_airbnb_paris.csv | Shape: (18350, 42)
row_id unique: True | NaN: 0
